# 04 — Results Analysis & Paper Figures
Generate all final tables and figures for the report.


In [ ]:
import os, json, numpy as np, pandas as pd
import matplotlib.pyplot as plt
if not os.path.exists('train.py'): os.chdir('occlusion-robust-grasp-cornell')

FIGS = 'results/paper_figures'
os.makedirs(FIGS, exist_ok=True)

In [ ]:
# Load metrics
with open('results/comparison/metrics.json') as f:
    metrics = json.load(f)

for name, m in metrics.items():
    print(f"{name}: {m['standard_accuracy']*100:.1f}% ({m['n_correct']}/{m['n_total']})")

In [ ]:
# Table 1: Standard accuracy
rows = []
for name, m in metrics.items():
    rows.append({'Model': name,
                 'Accuracy (%)': round(m['standard_accuracy']*100, 1),
                 'Correct': m['n_correct'],
                 'Total': m['n_total']})
df1 = pd.DataFrame(rows).set_index('Model')
print('Table 1: Standard Grasp Accuracy (IoU>0.25, angle<30°)')
display(df1)
df1.to_csv(f'{FIGS}/table1_accuracy.csv')

In [ ]:
# Table 2: Occlusion sweep
rows2 = []
for name, m in metrics.items():
    row = {'Model': name}
    sweep = m.get('occlusion_sweep', {})
    for frac in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6]:
        key = str(round(frac, 2))
        v   = sweep.get(key, sweep.get(frac, None))
        row[f'{int(frac*100)}% occ.'] = round(v*100, 1) if v is not None else '-'
    rows2.append(row)
df2 = pd.DataFrame(rows2).set_index('Model')
print('Table 2: Accuracy vs Occlusion Level (%)')
display(df2)
df2.to_csv(f'{FIGS}/table2_occlusion_sweep.csv')

In [ ]:
# Figure 1: Occlusion curve (high quality)
import matplotlib
matplotlib.rcParams.update({'font.size': 12, 'figure.dpi': 150})

fig, ax = plt.subplots(figsize=(8, 5))
colours = ['#e74c3c', '#2ecc71']
markers = ['o', 's']

for (name, m), col, mk in zip(metrics.items(), colours, markers):
    sweep = {float(k): v for k, v in m.get('occlusion_sweep', {}).items()}
    x = sorted(sweep.keys())
    y = [sweep[k]*100 for k in x]
    ax.plot(x, y, marker=mk, color=col, linewidth=2.5,
            markersize=8, label=name)

ax.set_xlabel('Occlusion fraction (proportion of pixels masked)')
ax.set_ylabel('Grasp accuracy (%)')
ax.set_title('Grasp Accuracy vs. Occlusion Level\n(Cornell dataset, IoU>0.25, angle error<30°)')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.4)
ax.set_ylim(0, 105)
plt.tight_layout()
plt.savefig(f'{FIGS}/fig1_occlusion_curve.pdf')
plt.savefig(f'{FIGS}/fig1_occlusion_curve.png', dpi=150)
plt.show()
print('Saved fig1')

In [ ]:
# Figure 2: Degradation bar chart
names  = list(metrics.keys())
acc_0  = [metrics[n].get('occlusion_sweep', {}).get('0.0', metrics[n]['standard_accuracy'])*100 for n in names]
acc_30 = [metrics[n].get('occlusion_sweep', {}).get('0.3', 0)*100 for n in names]
acc_60 = [metrics[n].get('occlusion_sweep', {}).get('0.6', 0)*100 for n in names]

x  = np.arange(len(names))
w  = 0.25
fig, ax = plt.subplots(figsize=(8, 5))
b1 = ax.bar(x - w, acc_0,  w, label='No occlusion (0%)',  color='#2ecc71')
b2 = ax.bar(x,     acc_30, w, label='30% occluded',       color='#f39c12')
b3 = ax.bar(x + w, acc_60, w, label='60% occluded',       color='#e74c3c')

for bars in [b1, b2, b3]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h+0.5,
                f'{h:.1f}%', ha='center', va='bottom', fontsize=9)

ax.set_ylabel('Accuracy (%)')
ax.set_title('Accuracy Degradation Under Synthetic Occlusion')
ax.set_xticks(x); ax.set_xticklabels(names)
ax.legend(); ax.set_ylim(0, 115)
plt.tight_layout()
plt.savefig(f'{FIGS}/fig2_degradation_bar.pdf')
plt.savefig(f'{FIGS}/fig2_degradation_bar.png', dpi=150)
plt.show()
print('Saved fig2')

In [ ]:
print('All paper figures:')
for f in sorted(os.listdir(FIGS)):
    print(' ', f)